# Figure 3B alternative — public-dataset XIST bar plot

This is a bar-plot alternative to the retained Figure 3B violin plot. It summarizes verified 10,000-library-size-normalized single-cell XIST counts at gestational weeks 3 and 4. Bars show the cell-level mean and error bars show SEM; no inferential test is applied because cells are not independent biological replicates.

## 1. Configuration
Edit paths, stage labels, colors, or dimensions here.

In [ ]:
figure3_dir <- "."
source_file <- file.path(
  figure3_dir,
  "source_files",
  "Fig3B_public_XIST_normalized_counts.csv"
)
output_dir <- "generated_figures/Fig3B"
dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)

stage_levels <- c("W3-1", "W4-1")
stage_labels <- c("W3", "W4")

stage_fill_colors <- c(
  "W3-1" = "#F8766D",
  "W4-1" = "#00BFC4"
)
stage_outline_colors <- c(
  "W3-1" = "#C34F48",
  "W4-1" = "#008D95"
)

figure_width <- 3
figure_height <- 4.5

## 2. Load packages

In [ ]:
library(ggplot2)
library(dplyr)

## 3. Load and validate normalized XIST counts
The source CSV was extracted from `GSE155121_qc_raw_and_normalized.h5ad`, using the `normalized_counts` layer and the XIST feature.

In [ ]:
xist_cells <- read.csv(source_file, check.names = FALSE) %>%
  mutate(Week_stage = factor(Week_stage, levels = stage_levels))

stopifnot(
  all(c("Cell", "Week_stage", "XIST_normalized_count") %in%
    names(xist_cells)),
  !anyNA(xist_cells$Week_stage),
  !anyNA(xist_cells$XIST_normalized_count),
  all(xist_cells$XIST_normalized_count >= 0)
)

table(xist_cells$Week_stage)

## 4. Calculate bar heights and SEM
These summaries describe the distribution of cells within each stage; they are not biological-replicate estimates.

In [ ]:
xist_summary <- xist_cells %>%
  group_by(Week_stage) %>%
  summarise(
    n_cells = n(),
    mean = mean(XIST_normalized_count),
    sd = sd(XIST_normalized_count),
    sem = sd / sqrt(n_cells),
    .groups = "drop"
  )

xist_summary

## 5. Plot
The filled bars, outlines, black error bars, dimensions, and classic theme follow the visual language of Figure 3C.

In [ ]:
xist_bar_plot <- ggplot(
  xist_summary,
  aes(x = Week_stage, y = mean, fill = Week_stage, color = Week_stage)
) +
  geom_col(
    width = 0.55,
    alpha = 0.6,
    linewidth = 0.8
  ) +
  geom_errorbar(
    aes(ymin = mean - sem, ymax = mean + sem),
    width = 0.16,
    linewidth = 0.8,
    color = "black"
  ) +
  scale_fill_manual(values = stage_fill_colors, guide = "none") +
  scale_color_manual(values = stage_outline_colors, guide = "none") +
  scale_x_discrete(labels = stage_labels) +
  scale_y_continuous(
    limits = c(0, NA),
    breaks = seq(0, 4, by = 1),
    expand = expansion(mult = c(0, 0.06))
  ) +
  labs(
    x = NULL,
    y = "Normalized XIST counts"
  ) +
  theme_classic(base_size = 14) +
  theme(
    plot.title = element_blank(),
    axis.title.y = element_text(size = 17),
    axis.text.x = element_text(size = 16, color = "black"),
    axis.text.y = element_text(size = 14, color = "black"),
    axis.line = element_line(linewidth = 0.6, color = "black"),
    axis.ticks = element_line(linewidth = 0.6, color = "black"),
    axis.ticks.length = unit(5, "pt"),
    panel.grid = element_blank()
  )

xist_bar_plot

## 6. Export

In [ ]:
output_file <- file.path(
  output_dir,
  "Fig3B_XIST_public_dataset_bar.svg"
)
grDevices::svg(
  output_file,
  width = figure_width,
  height = figure_height
)
print(xist_bar_plot)
grDevices::dev.off()

stopifnot(file.exists(output_file))
file.info(output_file)[, c("size", "mtime")]